In [ ]:
# ===== Colab Setup =====

REPO_URL = "https://github.com/dariuscparker/Self-Improving-Agent-Projects.git"
REPO_ROOT = "/content/Self-Improving-Agent-Projects"
WORKDIR = f"{REPO_ROOT}/modules/01_test_time_compute"

import os
import sys
import subprocess

# Remove any old copy (optional)
if os.path.exists(REPO_ROOT):
    subprocess.run(["rm", "-rf", REPO_ROOT], check=True)

# Clone fresh repo
subprocess.run(["git", "clone", REPO_URL, REPO_ROOT], check=True)

# Verify expected directories exist
if not os.path.isdir(REPO_ROOT):
    raise FileNotFoundError(f"Repo root not found: {REPO_ROOT}")

if not os.path.isdir(WORKDIR):
    raise FileNotFoundError(f"Expected notebook directory not found: {WORKDIR}")

# Add repo root to Python path
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

# Move into working directory
os.chdir(WORKDIR)

# Verify
print("Current working directory:", os.getcwd())
print("Top-level files here:", os.listdir(".")[:20])

Current working directory: /content/Self-Improving-Agent-Projects/modules/01_test_time_compute
Top-level files here: ['hw1-test-time-compute.ipynb']


# HW: Test-Time Compute Strategies on GSM8K

This notebook reflects the exercises generated by ChatGPT to review the course material in module one of Stanford's CS239A course. In this notebook, we will explore various test time compute strategies to help improve the results of a small model.

The first module illustrates that 'more compute' is typically associated with better performance. Using strategies to increase prompting can help our end result.

We will use the **GSM8K** dataset and a **lightweight open model** (**Qwen/Qwen2.5-1.5B-Instruct**) with 4-bit quantization. We will show how a lightweight model can achieve improved results with test time compute strategies. We will explore the following techniques in this notebook:

- **Part 1:** Single-pass CoT baseline (System-1)
- **Part 2:** Best-of-N sampling + **majority vote** (self-consistency)
- **Part 3:** Best-of-N + **simple verifier** (Proposer–Verifier using labels)
- **Part 4:** **Difficulty binning** (Easy/Medium/Hard) via empirical pass rate
- **Part 5:** **Sequential revisions** (self-refinement chains)
- **Part 6:** Lightweight **beam/search** over partial reasoning
- **Part 7:** Difficulty-aware **routing policy** (compute-optimal TTC) quick

In [ ]:
# REPO_URL = "https://github.com/dariuscparker/Self-Improving-Agent-Projects.git"
# REPO_DIR = "/content/Self-Improving-Agent-Projects"

# import os, sys

# %cd /content

# if not os.path.exists(REPO_DIR):
#     !git clone {REPO_URL} {REPO_DIR}
# else:
#     %cd {REPO_DIR}
#     !git pull

# if REPO_DIR not in sys.path:
#     sys.path.insert(0, REPO_DIR)

# %cd /content/Self-Improving-Agent-Projects/modules/01_test_time_compute
# !pwd
# !ls

## Part 0 — Setup
Set up the environment, load GSM8K and load a lightweight model. For this assignment, we will use Qwen2.5-1.5B Instruct. More information can be found about the model at the link below.

https://huggingface.co/Qwen/Qwen2.5-1.5B-Instruct



In [ ]:
!pip -q install -U bitsandbytes transformers accelerate datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 12.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 65.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 19.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 11.9 MB/s eta 0:00:00


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from datasets import load_dataset

import torch
import re
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from tqdm import tqdm
from collections import Counter

import time

# Global timer start
NOTEBOOK_START_TIME = time.time()


### Shared Helper Utilities

To keep this notebook readable and modular, reusable utility functions are stored in:

`shared/helpers.py`

This helper module contains common logic used throughout the notebook, including:

- answer extraction
- example display helpers
- single-generation utilities
- majority voting
- evaluation logic for both the baseline and self-consistency settings

In [ ]:
# ## Shared Helper Functions
import sys
from pathlib import Path
import importlib

# Locate repo root by finding shared/helpers.py
_repo_root = next(
    (p for p in [Path.cwd(), *Path.cwd().parents] if (p / "shared" / "helpers.py").exists()),
    None
)

if _repo_root is None:
    raise FileNotFoundError("Could not locate shared/helpers.py from the current notebook directory.")

if str(_repo_root) not in sys.path:
    sys.path.insert(0, str(_repo_root))

import shared.helpers as helpers
importlib.reload(helpers)

# Pull commonly used names into notebook scope
final_re = helpers.final_re
extract_true_answer = helpers.extract_true_answer
extract_pred_answer = helpers.extract_pred_answer
show_example = helpers.show_example
generate_one = helpers.generate_one
majority_vote = helpers.majority_vote
evaluate = helpers.evaluate
build_generation_pool = helpers.build_generation_pool
evaluate_from_generation_pool = helpers.evaluate_from_generation_pool
evaluate_best_of_n_simple_verifier = helpers.evaluate_best_of_n_simple_verifier
build_difficulty_table_from_pool = helpers.build_difficulty_table_from_pool

# New Part 5 helpers
build_revision_prompt = helpers.build_revision_prompt
run_revision_chain = helpers.run_revision_chain
select_revision_attempt = helpers.select_revision_attempt
evaluate_revision_chains = helpers.evaluate_revision_chains

### Load the GSM8K dataset and create train/eval subsets.

In [ ]:
# ## Load and Prepare Dataset
dataset = load_dataset("gsm8k", "main")
print("Dataset loaded successfully!")

# Create subsets (adjust sizes if you want)
train_subset = dataset["train"].select(range(1000))
eval_subset  = dataset["test"].select(range(500))

train_df = train_subset.to_pandas()
eval_df  = eval_subset.to_pandas()

print(f"Training subset: {len(train_df)} examples")
print(f"Evaluation subset: {len(eval_df)} examples")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

main/train-00000-of-00001.parquet:   0%|          | 0.00/2.31M [00:00<?, ?B/s]

main/test-00000-of-00001.parquet:   0%|          | 0.00/419k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/7473 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1319 [00:00<?, ? examples/s]

Dataset loaded successfully!
Training subset: 1000 examples
Evaluation subset: 500 examples


## Explore the Data

The GSM8K is a collection of middle school math problems that include questions and answers. It is a common benchmark data set for assessing reasoning models.

First, we can convert the dataset to pandas DataFrames for quick structural inspection.

In [ ]:
train_df = pd.DataFrame(dataset["train"])
test_df = pd.DataFrame(dataset["test"])
print("GSM8K train size:", len(train_df))
print("GSM8K test size:", len(test_df))
# Show the first five rows
train_df.head()

GSM8K train size: 7473
GSM8K test size: 1319


,question,answer
0,Natalia sold clips to 48 of her friends in Apr...,Natalia sold 48/2 = <<48/2=24>>24 clips in May...
1,Weng earns $12 an hour for babysitting. Yester...,Weng earns 12/60 = $<<12/60=0.2>>0.2 per minut...
2,Betty is saving money for a new wallet which c...,"In the beginning, Betty has only 100 / 2 = $<<..."
3,"Julie is reading a 120-page book. Yesterday, s...",Maila read 12 x 2 = <<12*2=24>>24 pages today....
4,James writes a 3-page letter to 2 different fr...,He writes each friend 3*2=<<3*2=6>>6 pages a w...


We can inspect a few training examples to confirm the question/answer format.

In [ ]:
# Use show_example function from shared/helpers.py to inspect the first three training examples.
for i in range(3):
    show_example(dataset["train"][i])



QUESTION:
 Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?

SOLUTION:
 Natalia sold 48/2 = <<48/2=24>>24 clips in May.
Natalia sold 48+24 = <<48+24=72>>72 clips altogether in April and May.
#### 72

QUESTION:
 Weng earns $12 an hour for babysitting. Yesterday, she just did 50 minutes of babysitting. How much did she earn?

SOLUTION:
 Weng earns 12/60 = $<<12/60=0.2>>0.2 per minute.
Working 50 minutes, she earned 0.2 x 50 = $<<0.2*50=10>>10.
#### 10

QUESTION:
 Betty is saving money for a new wallet which costs $100. Betty has only half of the money she needs. Her parents decided to give her $15 for that purpose, and her grandparents twice as much as her parents. How much more money does Betty need to buy the wallet?

SOLUTION:
 In the beginning, Betty has only 100 / 2 = $<<100/2=50>>50.
Betty's grandparents gave her 15 * 2 = $<<15*2=30>>30.
This means, Betty needs 100 - 50 - 30 

## Answer Pattern and Regex

It is important to understand the structure of a data set so that a reasoning model can be appropriately prompted. GSM8K solutions end with a standardized final-answer line:

- Final numeric answer appears after `####`
- It is on its own line
- It may be an integer or decimal


Because this format is consistent, we can create a regex pattern to reliably extract the true answer with a regular expression. The regex pattern below is defined in shared/helpers.py:

`final_re = re.compile(r"####\s*([-+]?\d+(?:\.\d+)?)")
`


In [ ]:
# Regex is defined in shared/helpers.py
print("Final answer regex:", final_re.pattern)


Final answer regex: ####\s*([-+]?\d+(?:\.\d+)?)


Also, notice that each solution not only includes the final answer, but includes reasoning that lead to the final answer. We defined `extract_true_answer` to parse each full solution and keep only the numeric final answer (the value after `####`.)

In [ ]:
dataset = dataset.map(lambda x: {"true_answer": extract_true_answer(x)})


Map:   0%|          | 0/7473 [00:00<?, ? examples/s]

Map:   0%|          | 0/1319 [00:00<?, ? examples/s]

Verify that the extracted `true_answer` aligns with the original `answer` text.


In [ ]:
for i in range(3):
    show_example(dataset["train"][i])



QUESTION:
 Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?

SOLUTION:
 Natalia sold 48/2 = <<48/2=24>>24 clips in May.
Natalia sold 48+24 = <<48+24=72>>72 clips altogether in April and May.
#### 72

TRUE ANSWER: 72

QUESTION:
 Weng earns $12 an hour for babysitting. Yesterday, she just did 50 minutes of babysitting. How much did she earn?

SOLUTION:
 Weng earns 12/60 = $<<12/60=0.2>>0.2 per minute.
Working 50 minutes, she earned 0.2 x 50 = $<<0.2*50=10>>10.
#### 10

TRUE ANSWER: 10

QUESTION:
 Betty is saving money for a new wallet which costs $100. Betty has only half of the money she needs. Her parents decided to give her $15 for that purpose, and her grandparents twice as much as her parents. How much more money does Betty need to buy the wallet?

SOLUTION:
 In the beginning, Betty has only 100 / 2 = $<<100/2=50>>50.
Betty's grandparents gave her 15 * 2 = $<<15*2=30>>30.
This

## Model Setup

In this section, we load a lightweight instruction-tuned language model:

- **Model:** Qwen2.5-1.5B-Instruct  
- **Type:** Causal Language Model (decoder-only transformer)  
- **Use case:** Step-by-step reasoning and instruction following  

This model is intentionally small (~1.5B parameters) so it can run efficiently in a Colab environment while still demonstrating reasoning capabilities. We will use Quantitization which significantly reduces GPU usage without a severe cost to precision.  We will use **4-bit quantization** via the `bitsandbytes` library.


### Check GPU Availability

In [ ]:
# Check GPU Availability
print("CUDA Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("Compute Capability:", torch.cuda.get_device_capability(0))


CUDA Available: True
GPU: Tesla T4
Compute Capability: (7, 5)


### Load the model (4-bit quantized)

In [ ]:
# Load Model with 4-bit Quantization (Lightweight)
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

model_name = "Qwen/Qwen2.5-1.5B-Instruct"
#model_name = "Qwen/Qwen2.5-0.5B-Instruct"


tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=quantization_config,
    device_map="auto",
    torch_dtype=torch.bfloat16
)

model.eval()
print("Model loaded:", model_name)


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model loaded: Qwen/Qwen2.5-1.5B-Instruct


### Part 1 — Run Baseline

The baseline is the simplest evaluation setting: the model produces one deterministic response per question, and that single extracted answer is scored directly.

Here we evaluate the model on a subset of the GSM8K evaluation set using:

- `n_samples=1`
- `temperature=0.0`

This gives us a reference point before adding extra inference-time compute through self-consistency.

In [ ]:
# ## Run Baseline (System-1)
# Tip: with the 1.5B model, 100 questions should usually be much faster.

baseline_metrics, baseline_results = evaluate(
    eval_df=eval_df,
    model=model,
    tokenizer=tokenizer,
    n_samples=1,
    temperature=0.0,
    max_questions=30
)

# Extract key values
baseline_acc = baseline_metrics["accuracy"]
baseline_q = baseline_metrics["num_questions"]
baseline_correct = int(baseline_acc * baseline_q)

print("Baseline metrics:")
print(json.dumps(baseline_metrics, indent=2))

print("\nSummary:")
print(f"Correct: {baseline_correct} / {baseline_q}")
print(f"Accuracy: {baseline_acc:.2%}")

100%|██████████| 30/30 [05:55<00:00, 11.86s/it]

Baseline metrics:
{
  "accuracy": 0.26666666666666666,
  "num_questions": 30,
  "n_samples": 1,
  "temperature": 0.0,
  "elapsed_sec": 355.72444200515747,
  "avg_input_tokens_per_question": 89.5,
  "avg_new_tokens_per_question": 171.8,
  "avg_total_tokens_per_question": 261.3
}

Summary:
Correct: 8 / 30
Accuracy: 26.67%


The baseline model answered **8 out of 30 questions correctly**, achieving an accuracy of **26.67%**.

This provides a simple reference point before introducing additional inference-time compute through multi-sample generation and alternative selection strategies.

## Parts 2 and 3 — Shared Proposal Pool

To compare **majority vote** and **verifier-based selection** more fairly, we first generate a **single frozen pool of candidate solutions** for each question.

In this setup, each question is sampled **5 times** with `temperature=0.7`, producing a shared set of proposals that is reused across both methods:

- **Part 2:** select the final answer using **majority vote**
- **Part 3:** select the final answer using a **simple verifier**

This is a more controlled experimental design than regenerating candidates separately for each section, because any difference in performance is more directly attributable to the **selection rule** rather than to different sampled generations.

In [ ]:
# Build one shared proposal pool for Parts 2 and 3
proposal_pool, proposal_pool_meta = build_generation_pool(
    eval_df=eval_df,
    model=model,
    tokenizer=tokenizer,
    n_samples=5,
    temperature=0.7,
    max_questions=30,
    max_new_tokens=256,
)

print("Shared proposal pool built.")
print(json.dumps(proposal_pool_meta, indent=2))

### Part 2 — Self-Consistency from a Shared Proposal Pool

Self-consistency increases inference-time compute by generating multiple candidate solutions for each question and selecting the most common final answer using majority vote.

Here, the candidate solutions are not regenerated inside Part 2. Instead, they come from the shared frozen proposal pool built above. This means the model still uses **5 sampled generations per question**, but those same generations will also be reused in Part 3.

The key idea behind self-consistency is that randomness (`temperature=0.7`) allows the model to explore different reasoning paths. Each generated response is reduced to a final numeric answer, and the most frequently occurring answer is selected as the final prediction.

For example, if the five sampled answers are `[18, 18, 22, 18, 22]`, the final prediction is **18** based on majority vote.

Using a shared pool makes this method easier to compare fairly against verifier-based selection, since both methods operate on the exact same candidate set.

In [ ]:
# Part 2: Self-Consistency from the shared proposal pool
sc_metrics, sc_results = evaluate_from_generation_pool(
    generation_pool=proposal_pool,
    pool_meta=proposal_pool_meta,
    selection_mode="majority_vote",
)

sc_acc = sc_metrics["accuracy"]
sc_q = sc_metrics["num_questions"]
sc_correct = sc_metrics["num_correct"]

print("Self-consistency metrics:")
print(json.dumps(sc_metrics, indent=2))

print("\nSummary:")
print(f"Correct: {sc_correct} / {sc_q}")
print(f"Accuracy: {sc_acc:.2%}")

 70%|███████   | 21/30 [18:16<07:49, 52.20s/it]


KeyboardInterrupt: 

### Part 2 Results Summary

Using the shared proposal pool, self-consistency selected the final answer by applying **majority vote** to the same five sampled candidates for each question.

Because the candidate set was frozen in advance, this experiment isolates the effect of the **majority-vote selector** rather than mixing selection effects with newly sampled generations.

## Part 3 — Simple Verifier (Proposer–Verifier Using Labels)

In this section, we compare majority voting with a simple verifier-based selection strategy using the **same frozen proposal pool** from Part 2.

The procedure is:
1. Reuse the same sampled candidate solutions generated earlier
2. Score each candidate with a simple verifier based on the known GSM8K answer
3. Select the candidate with the highest verifier score

This approximates an outcome-based proposer–verifier setup: instead of choosing the most common answer across samples, we choose the single candidate that best matches the label according to the verifier.

### Run Part 3 (Best-of-N + verifier)

In [ ]:
# Part 3: Simple verifier from the same shared proposal pool
verifier_metrics, verifier_results = evaluate_from_generation_pool(
    generation_pool=proposal_pool,
    pool_meta=proposal_pool_meta,
    selection_mode="simple_verifier",
    verifier_mode="exact",   # can also try "numeric_distance"
)

verifier_acc = verifier_metrics["accuracy"]
verifier_q = verifier_metrics["num_questions"]
verifier_correct = verifier_metrics["num_correct"]

print("Verifier metrics:")
print(json.dumps(verifier_metrics, indent=2))

print("\nSummary:")
print(f"Correct: {verifier_correct} / {verifier_q}")
print(f"Accuracy: {verifier_acc:.2%}")

## Part 4 — Difficulty Binning (Easy / Medium / Hard)

In this section, we estimate empirical problem difficulty using a small number of stochastic generations per question.

For each question, we:
1. sample the model multiple times with mild randomness,
2. compute the fraction of samples that produce the correct final answer, and
3. assign a difficulty bin based on that empirical pass rate.

We use the following bins:

- **Easy:** pass rate ≥ 0.8  
- **Medium:** 0.3 ≤ pass rate < 0.8  
- **Hard:** pass rate < 0.3

This gives a simple way to approximate which problems are consistently solved by the model and which remain unstable or difficult.

After assigning bins, we compare baseline, self-consistency, and simple-verifier performance across those difficulty groups.

In [ ]:
difficulty_df = build_difficulty_table_from_pool(proposal_pool)

print("Difficulty bin counts:")
print(difficulty_df["difficulty"].value_counts())

difficulty_df.head()

### Difficulty Binning Summary

Difficulty here is defined relative to the model’s own empirical pass rate on each problem.

Because we reuse the same shared proposal pool from Parts 2 and 3, these bins are based on the same sampled candidate sets used in the earlier experiments. This keeps the notebook more consistent and avoids unnecessary extra generation cost.

A high pass rate suggests that the problem is reliably solved by the model, while a low pass rate suggests that the problem remains unstable or difficult even when multiple reasoning paths are sampled.

## Part 5 — Sequential Revisions (Self-Refinement)

In this section, we test whether a model can improve its own answer by revising earlier reasoning.

Instead of sampling several independent solutions at once, we build a **revision chain**:

- **Attempt 0:** baseline solution
- **Attempts 1..R:** revised solutions conditioned on the previous attempt

This gives a different form of test-time compute. Rather than increasing breadth through parallel samples, we increase depth through iterative self-correction.

We evaluate several ways to choose a final answer from the chain:

- **Last revision:** use the final attempt directly
- **Majority vote:** choose the most common predicted answer across all attempts
- **Simple verifier:** choose the attempt with the highest verifier score

In [ ]:
# Reuse the existing proposal_pool from Parts 2 and 3
#difficulty_df = build_difficulty_table_from_pool(proposal_pool)

# Attach difficulty labels to the same eval subset used for the pool
eval_with_diff = eval_df.iloc[:len(difficulty_df)].copy().reset_index().rename(columns={"index": "idx"})
eval_with_diff = eval_with_diff.merge(
    difficulty_df[["idx", "difficulty", "pass_rate"]],
    on="idx",
    how="left"
)

eval_with_diff[["idx", "difficulty", "pass_rate"]].head()

In [ ]:
rev_last_metrics, rev_last_rows = evaluate_revision_chains(
    eval_df=eval_with_diff,
    model=model,
    tokenizer=tokenizer,
    r_steps=3,
    base_temperature=0.0,
    rev_temperature=0.7,
    max_questions=5,
    max_new_tokens=256,
    selection_mode="last",
)

print("Revision metrics (last):")
print(json.dumps(rev_last_metrics, indent=2))

In [ ]:
rev_mv_metrics, rev_mv_rows = evaluate_revision_chains(
    eval_df=eval_with_diff,
    model=model,
    tokenizer=tokenizer,
    r_steps=3,
    base_temperature=0.0,
    rev_temperature=0.7,
    max_questions=5,
    max_new_tokens=256,
    selection_mode="majority_vote",
)

print("Revision metrics (majority vote):")
print(json.dumps(rev_mv_metrics, indent=2))

In [ ]:
rev_ver_metrics, rev_ver_rows = evaluate_revision_chains(
    eval_df=eval_with_diff,
    model=model,
    tokenizer=tokenizer,
    r_steps=3,
    base_temperature=0.0,
    rev_temperature=0.7,
    max_questions=5,
    max_new_tokens=256,
    selection_mode="simple_verifier",
    verifier_mode="exact",
)

print("Revision metrics (simple verifier):")
print(json.dumps(rev_ver_metrics, indent=2))

### Part 5 Results Summary

Sequential revisions test a different type of inference-time compute from Best-of-N sampling.

- **Best-of-N** explores several independent reasoning paths in parallel
- **Sequential revisions** repeatedly improves one evolving reasoning path

This makes Part 5 useful for comparing **breadth vs. depth** at test time.

If revisions help, that suggests the model can sometimes recognize and repair its own mistakes.
If revisions do not help, that suggests either:
- the initial mistake is persistent across revisions, or
- the model lacks a strong enough self-correction signal.

In the next step, we can compare revision performance against earlier methods under a roughly similar token budget.

## Part 6 — Lightweight Beam/Search

This is a **simple, teaching-friendly** beam-style search over *partial reasoning*:

- Keep a beam of **B** partial solutions
- Expand each partial solution with **M** candidate “next steps”
- Score partials using a heuristic (here: **average logprob per generated token** with a length penalty)
- Stop if a candidate emits a `Final Answer: ...` or after `max_steps`

Then select the best finished candidate using the verifier.

Because it’s compute-heavy, run on **100–200** examples.


In [ ]:
import re
import torch
from tqdm.auto import tqdm

# =========================
# Helper: Final Answer Extraction
# =========================

def extract_final_answer(text: str):
    """
    Beam / step prompts request:
      'Final Answer: <number>'
    Fallbacks to generic numeric extraction.
    """
    if text is None:
        return None
    s = str(text)

    m = re.search(r"Final Answer:\s*([-+]?\d[\d,]*(?:\.\d+)?)", s)
    if m:
        return m.group(1).replace(",", "").strip()

    return _extract_final_answer_from_text(s)


# =========================
# Beam Search Utilities
# =========================

def generate_continuations_with_scores(
    model,
    tokenizer,
    prompt_text,
    num_return_sequences=2,
    temperature=0.7,
    max_new_tokens=80
):
    """
    Generate multiple continuations and compute a heuristic avg log-prob score.
    Handles tokenizers where apply_chat_template returns either:
      - a dict-like BatchEncoding, or
      - a plain Tensor of input_ids
    """
    messages = [{"role": "user", "content": prompt_text}]
    device = getattr(model, "device", "cuda" if torch.cuda.is_available() else "cpu")

    # --- Robust encoding across tokenizer variants ---
    if hasattr(tokenizer, "apply_chat_template"):
        enc = tokenizer.apply_chat_template(
            messages,
            tokenize=True,
            return_tensors="pt"
        )
        if isinstance(enc, torch.Tensor):
            # Some tokenizers return input_ids tensor directly
            input_ids = enc.to(device)
            attention_mask = torch.ones_like(input_ids, device=device)
            enc = {"input_ids": input_ids, "attention_mask": attention_mask}
        else:
            # BatchEncoding / dict-like
            enc = {k: v.to(device) for k, v in enc.items()}
            if "attention_mask" not in enc:
                enc["attention_mask"] = torch.ones_like(enc["input_ids"], device=device)
    else:
        # Fallback: plain tokenization
        enc = tokenizer(prompt_text, return_tensors="pt")
        enc = {k: v.to(device) for k, v in enc.items()}
        if "attention_mask" not in enc:
            enc["attention_mask"] = torch.ones_like(enc["input_ids"], device=device)

    out = model.generate(
        **enc,
        max_new_tokens=max_new_tokens,
        do_sample=(temperature > 0),
        temperature=temperature,
        num_return_sequences=num_return_sequences,
        return_dict_in_generate=True,
        output_scores=True,
        pad_token_id=(tokenizer.pad_token_id if tokenizer.pad_token_id is not None else tokenizer.eos_token_id),
        eos_token_id=tokenizer.eos_token_id,
    )

    sequences = out.sequences
    scores_per_step = out.scores  # list[T] of (batch, vocab)

    results = []
    prompt_len = enc["input_ids"].shape[-1]

    for b in range(sequences.shape[0]):
        full_ids = sequences[b]
        gen_ids = full_ids[prompt_len:]
        gen_tokens = int(gen_ids.shape[-1])
        text = tokenizer.decode(full_ids, skip_special_tokens=True)

        avg_logprob = 0.0
        if gen_tokens > 0:
            logps = []
            for t in range(gen_tokens):
                step_scores = scores_per_step[t][b]
                step_logp = torch.log_softmax(step_scores, dim=-1)
                tok_id = int(gen_ids[t].item())
                logps.append(float(step_logp[tok_id].item()))
            avg_logprob = float(sum(logps) / max(1, len(logps)))

        results.append({
            "text": text,
            "gen_tokens": gen_tokens,
            "avg_logprob": avg_logprob,
            "final_answer": extract_final_answer(text),
        })

    return results



def make_step_prompt(question, partial_reasoning):
    return (
        "You are solving a math word problem.\n"
        "Extend the solution by ONE clear next step.\n"
        "If you are ready to finish, output on a new line exactly: Final Answer: <number>\n\n"
        f"Question:\n{question}\n\n"
        f"Current Partial Solution:\n{partial_reasoning}\n"
    )


# =========================
# Beam Search Core
# =========================

def beam_search_gsm8k(
    model,
    tokenizer,
    question,
    true_answer=None,
    beam_size=3,
    expansions_per_beam=2,
    max_steps=6,
    temperature=0.7,
    step_max_new_tokens=80,
    length_penalty=0.02,
    verifier_mode="exact",
):
    beam = [{
        "partial": "",
        "score": 0.0,
        "steps": 0,
        "tokens": 0,
        "finished": False,
        "final_answer": None,
    }]

    finished = []

    for step in range(max_steps):
        new_candidates = []

        for b in beam:
            if b["finished"]:
                new_candidates.append(b)
                continue

            prompt = make_step_prompt(question, b["partial"])
            conts = generate_continuations_with_scores(
                model=model,
                tokenizer=tokenizer,
                prompt_text=prompt,
                num_return_sequences=expansions_per_beam,
                temperature=temperature,
                max_new_tokens=step_max_new_tokens,
            )

            for c in conts:
                new_partial = c["text"]
                total_tokens = b["tokens"] + c["gen_tokens"]

                score = (
                    b["score"]
                    + c["avg_logprob"]
                    - (length_penalty * total_tokens)
                )

                is_finished = (
                    c["final_answer"] is not None
                    and str(c["final_answer"]).strip() != ""
                )

                new_candidates.append({
                    "partial": new_partial,
                    "score": score,
                    "steps": step + 1,
                    "tokens": total_tokens,
                    "finished": is_finished,
                    "final_answer": c["final_answer"],
                })

        beam = sorted(new_candidates, key=lambda x: x["score"], reverse=True)[:beam_size]

        for b in beam:
            if b["finished"]:
                finished.append(b)

        if len(finished) >= beam_size:
            break

    pool = finished if finished else beam

    if true_answer is not None:
        best = None
        best_vs = float("-inf")
        for p in pool:
            vs = verifier_score(p.get("final_answer", ""), true_answer, mode=verifier_mode)
            if vs > best_vs:
                best_vs = vs
                best = p
        best = dict(best)
        best["verifier_score"] = best_vs
        best["verifier_mode"] = verifier_mode
        best["used_pool"] = "finished" if finished else "beam"
        return best

    return pool[0]


# =========================
# Evaluation Wrapper
# =========================

def evaluate_beam_search(
    eval_df,
    model,
    tokenizer,
    beam_size=3,
    expansions_per_beam=2,
    max_steps=6,
    temperature=0.7,
    step_max_new_tokens=80,
    length_penalty=0.02,
    verifier_mode="exact",
):
    rows = []
    correct = 0
    total_tokens = 0

    for i, row in tqdm(
        eval_df.iterrows(),
        total=len(eval_df),
        desc=f"Beam search (B={beam_size})",
    ):
        q = row["question"]
        true = extract_gsm8k_final_answer(row["answer"])

        best = beam_search_gsm8k(
            model=model,
            tokenizer=tokenizer,
            question=q,
            true_answer=true,
            beam_size=beam_size,
            expansions_per_beam=expansions_per_beam,
            max_steps=max_steps,
            temperature=temperature,
            step_max_new_tokens=step_max_new_tokens,
            length_penalty=length_penalty,
            verifier_mode=verifier_mode,
        )

        pred = best.get("final_answer", "")
        is_correct = (str(pred).strip() == str(true).strip())

        correct += int(is_correct)
        total_tokens += int(best.get("tokens", 0))

        rows.append({
            "idx": int(row["idx"]) if "idx" in row else int(i),
            "difficulty": row.get("difficulty", None),
            "question": q,
            "true_answer": true,
            "pred_answer": pred,
            "correct": bool(is_correct),
            "tokens": int(best.get("tokens", 0)),
            "score": float(best.get("score", 0.0)),
            "verifier_score": best.get("verifier_score", None),
            "text": best.get("partial", ""),
        })

    metrics = {
        "method": "beam_search",
        "beam_size": beam_size,
        "expansions_per_beam": expansions_per_beam,
        "max_steps": max_steps,
        "temperature": temperature,
        "step_max_new_tokens": step_max_new_tokens,
        "length_penalty": length_penalty,
        "verifier_mode": verifier_mode,
        "accuracy": correct / max(1, len(eval_df)),
        "avg_tokens": total_tokens / max(1, len(eval_df)),
    }

    return metrics, rows


### Run Part 6 (Beam/Search)
Run on ~100 examples first (this can be slower than revisions).

In [ ]:
beam_metrics, beam_rows = evaluate_beam_search(
    eval_df=eval_with_diff.head(5),  # start small
    model=model,
    tokenizer=tokenizer,
    beam_size=3,
    expansions_per_beam=2,
    max_steps=6,
    temperature=0.7,
    step_max_new_tokens=80,
    length_penalty=0.02,
    verifier_mode="exact",
)

print(beam_metrics)


## Part 7 — Difficulty-Aware, Compute-Optimal Policy

We route questions by difficulty bin to different TTC strategies:

- **S1:** Baseline (single pass)
- **S2:** Best-of-N (small N) majority
- **S3:** Revisions (small R) + verifier selection
- **S4:** High compute (larger N or beam search)

You will compare policies like A/B/C against uniform strategies (always S2, always S3, etc.).


In [ ]:
from tqdm.auto import tqdm

# =========================
# Part 7 — Routing Policies
# =========================

def run_strategy(row, model, tokenizer, strategy_name, cfg):
    q = row["question"]
    true = row["answer"]

    if strategy_name == "S1":
        # If you don't have solve_gsm8k_question anymore, swap this to sample_solutions(..., n_samples=1)
        full_text, pred, toks = solve_gsm8k_question(
            model=model, tokenizer=tokenizer, question=q,
            temperature=cfg.get("temperature", 0.0),
            max_new_tokens=cfg.get("max_new_tokens", 256)
        )
        return {"pred": pred, "tokens": toks, "text": full_text}

    if strategy_name == "S2":
        sols = sample_solutions(
            model=model, tokenizer=tokenizer, question=q,
            n_samples=cfg.get("n_samples", 5),
            temperature=cfg.get("temperature", 0.7),
            max_new_tokens=cfg.get("max_new_tokens", 256)
        )
        answers = [str(s.get("final_answer","")).strip() for s in sols]
        counts = {}
        for a in answers:
            counts[a] = counts.get(a, 0) + 1
        best_answer = sorted(counts.items(), key=lambda x: (-x[1], x[0]))[0][0]

        chosen = None
        for s in sols:
            if str(s.get("final_answer","")).strip() == best_answer:
                chosen = s
                break

        total_tokens = sum([s.get("gen_tokens",0) for s in sols])
        return {
            "pred": chosen.get("final_answer","") if chosen else best_answer,
            "tokens": total_tokens,
            "text": chosen.get("full_text","") if chosen else ""
        }

    if strategy_name == "S3":
        chain = run_revision_chain(
            model=model, tokenizer=tokenizer, question=q,
            r_steps=cfg.get("r_steps", 2),
            base_temperature=cfg.get("base_temperature", 0.0),
            rev_temperature=cfg.get("rev_temperature", 0.7),
            max_new_tokens=cfg.get("max_new_tokens", 256)
        )
        chosen = select_from_attempts(
            chain, true_answer=true,
            selection=cfg.get("selection", "verifier"),
            verifier_mode=cfg.get("verifier_mode", "exact")
        )
        total_tokens = sum([a.get("gen_tokens",0) for a in chain])
        return {"pred": chosen.get("final_answer",""), "tokens": total_tokens, "text": chosen.get("full_text","")}

    if strategy_name == "S4":
        if cfg.get("mode","beam") == "beam":
            best = beam_search_gsm8k(
                model=model, tokenizer=tokenizer, question=q, true_answer=true,
                beam_size=cfg.get("beam_size", 3),
                expansions_per_beam=cfg.get("expansions_per_beam", 2),
                max_steps=cfg.get("max_steps", 6),
                temperature=cfg.get("temperature", 0.7),
                step_max_new_tokens=cfg.get("step_max_new_tokens", 80),
                length_penalty=cfg.get("length_penalty", 0.02),
                verifier_mode=cfg.get("verifier_mode", "exact")
            )
            return {"pred": best.get("final_answer",""), "tokens": int(best.get("tokens",0)), "text": best.get("partial","")}
        else:
            best = best_of_n_with_verifier(
                model=model, tokenizer=tokenizer, question=q, true_answer=true,
                n_samples=cfg.get("n_samples", 20),
                temperature=cfg.get("temperature", 0.7),
                max_new_tokens=cfg.get("max_new_tokens", 256),
                verifier_mode=cfg.get("verifier_mode", "exact")
            )
            # NOTE: best_of_n_with_verifier returns best candidate; token accounting can vary by your implementation
            return {"pred": best.get("final_answer",""), "tokens": best.get("gen_tokens",0), "text": best.get("full_text","")}

    raise ValueError(f"Unknown strategy: {strategy_name}")


def evaluate_policy(eval_df, model, tokenizer, policy_map, cfg_map, desc="Policy"):
    rows = []
    correct = 0
    total_tokens = 0

    for i, row in tqdm(eval_df.iterrows(), total=len(eval_df), desc=desc):
        diff = row.get("difficulty", "medium")
        strat = policy_map.get(diff, "S2")
        out = run_strategy(row, model, tokenizer, strat, cfg_map.get(strat, {}))

        pred = out["pred"]
        true = row["answer"]
        is_correct = (str(pred).strip() == str(true).strip())

        correct += int(is_correct)
        total_tokens += int(out["tokens"])

        rows.append({
            "idx": int(i),
            "difficulty": diff,
            "strategy": strat,
            "answer": true,
            "pred_answer": pred,
            "correct": bool(is_correct),
            "tokens": int(out["tokens"])
        })

    acc = correct / max(1, len(eval_df))
    avg_tokens = total_tokens / max(1, len(eval_df))
    return {"accuracy": acc, "avg_tokens": avg_tokens}, rows


# Example policy definitions (edit as desired)
policy_A = {"easy": "S1", "medium": "S2", "hard": "S3"}
policy_B = {"easy": "S1", "medium": "S3", "hard": "S4"}
policy_C = {"easy": "S2", "medium": "S3", "hard": "S4"}

cfg_map = {
    "S1": {"temperature": 0.0, "max_new_tokens": 256},
    "S2": {"n_samples": 5, "temperature": 0.7, "max_new_tokens": 256},
    "S3": {"r_steps": 2, "base_temperature": 0.0, "rev_temperature": 0.7, "max_new_tokens": 256, "selection":"verifier", "verifier_mode":"exact"},
    "S4": {"mode":"beam", "beam_size": 3, "expansions_per_beam": 2, "max_steps": 6, "temperature": 0.7, "step_max_new_tokens": 80, "length_penalty": 0.02, "verifier_mode":"exact"}
}

# Evaluate on a subset first (with progress bars)
subset = eval_with_diff.head(120)

polA_metrics, polA_rows = evaluate_policy(subset, model, tokenizer, policy_A, cfg_map, desc="Policy A")
polB_metrics, polB_rows = evaluate_policy(subset, model, tokenizer, policy_B, cfg_map, desc="Policy B")
polC_metrics, polC_rows = evaluate_policy(subset, model, tokenizer, policy_C, cfg_map, desc="Policy C")

print("Policy A:", polA_metrics)
print("Policy B:", polB_metrics)
print("Policy C:", polC_metrics)

### Save results

In [ ]:
# ## Save Results

final_output = {
    "model_name": model_name,
    "baseline_metrics": baseline_metrics,
    "baseline_results": baseline_results,
    "self_consistency_metrics": sc_metrics,
    "self_consistency_results": sc_results,
}

with open("gsm8k_refactored_results.json", "w") as f:
    json.dump(final_output, f, indent=2)

print("Saved: gsm8k_refactored_results.json")


## Results Summary + Plots

Fill this section with your final runs (200–500 eval questions).  
Minimum plots/tables to include in your report:

- Accuracy vs. Avg tokens for:
  - Baseline (Part 1)
  - Best-of-N majority (Part 2)
  - Best-of-N verifier (Part 3)
  - Revisions (Part 5)
  - Beam/search (Part 6; on a subset is fine)
- Accuracy by difficulty bin (Easy/Medium/Hard)
- Policy routing comparison (Part 7)


In [ ]:
# Collect the metrics you ran above into one DataFrame for quick comparison.
metric_rows = []

# These variables exist if you ran the example cells; add/remove as needed.
for m in [baseline_metrics, sc_metrics, bover_metrics, rev_metrics, beam_metrics]:
    if isinstance(m, dict):
        metric_rows.append(m)

metrics_df = pd.DataFrame(metric_rows)
metrics_df


In [ ]:
# Simple plot: accuracy vs avg_tokens
if "accuracy" in metrics_df.columns and "avg_tokens" in metrics_df.columns:
    plt.figure()
    plt.scatter(metrics_df["avg_tokens"], metrics_df["accuracy"])
    for _, r in metrics_df.iterrows():
        label = r.get("method","")
        plt.annotate(label, (r["avg_tokens"], r["accuracy"]))
    plt.xlabel("Avg tokens (TTC proxy)")
    plt.ylabel("Accuracy")
    plt.title("Accuracy vs Test-Time Compute (tokens)")
    plt.show()


## Report Template (3–5 pages)

Use this structure (you can paste into a Google Doc or write directly here):

1. **Motivation**
   - What is test-time compute (TTC)?
   - Why GSM8K and why a lightweight open model?
   - What you expect: sampling, selection, revisions, and search trade compute for accuracy.

2. **Methods**
   - Dataset split sizes and preprocessing
   - Model details (Qwen2.5-1.5B, quantized), decoding settings
   - Implemented strategies (Parts 1–7), with key hyperparameters (N, R, B, M, etc.)
   - Compute metric: average generated tokens (and/or wall-clock, if recorded)

3. **Results**
   - Overall accuracy vs compute tradeoff for each method
   - Breakdown by difficulty bin (Easy/Medium/Hard)
   - Policy routing comparison vs uniform strategies

4. **Discussion**
   - Where did TTC help most (which difficulty bins)?
   - Majority vote vs verifier selection: when does each win?
   - Revisions vs Best-of-N under similar compute budgets
   - Is beam/search “worth it” for this model/budget?
   - Connection back to the behaviors described in Snell-style TTC work

5. **Limitations**
   - Token proxy vs true FLOPs/time
   - Heuristic scoring for beam search (no learned PRM)
   - Small model constraints and prompt sensitivity


In [ ]:
# Optional: export your metrics and selected rows to JSON for your report
out = {
    "metrics_table": metrics_df.to_dict(orient="records"),
    "policy_A": polA_metrics,
    "policy_B": polB_metrics,
    "policy_C": polC_metrics,
}
with open("hw_results_summary.json","w") as f:
    json.dump(out, f, indent=2)
print("Wrote hw_results_summary.json")


In [ ]:
NOTEBOOK_END_TIME = time.time()
total_runtime = NOTEBOOK_END_TIME - NOTEBOOK_START_TIME

print("\n===== TOTAL NOTEBOOK RUNTIME =====")
print("Total time (seconds):", round(total_runtime, 2))
print("Total time (minutes):", round(total_runtime / 60, 2))